In [ ]:
import json
import torch
from random import randint
from utils.plotter import drawQubitAllocation
from sampler.hardwaresampler import HardwareSampler
from sampler.randomcircuit import RandomCircuit
from sqarl.sqarl import SQARL, ModelConfigs
from utils.customtypes import Hardware, Circuit

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# How to load a trained model and optimize a circuit

In [ ]:
sqarl = SQARL.load("../trained/da_v2_ft", device=device).set_mode(SQARL.Mode.Sequential) # Load last checkpoint in sequential allocation mode


# Load the dataset of benchmark circuits...
with open(f'../data/circuits_50.json', 'r') as f:
  data = json.load(f)

n_qubits = data['n_qubits']
circuit_slices = data['circuits']['deutsch_jozsa']
circuit = Circuit(slice_gates=circuit_slices, n_qubits=n_qubits)

# ... or generate a random circuit
circuit = RandomCircuit(num_lq=50, num_slices=50).sample()

# Specify a hardware configuration
n_cores = 5
hardware = Hardware(
  core_capacities=torch.tensor([10]*n_cores),
  core_connectivity=(torch.ones(size=(n_cores,n_cores)) - torch.eye(n_cores)),
)

allocs, cost = sqarl.optimize(circuit, hardware=hardware, verbose=True)

print(f"\nFinal cost is: {cost}")

# Might blow up if circuit is too large due to ram
drawQubitAllocation(
  qubit_allocation=allocs,
  core_capacities=hardware.core_capacities,
  circuit_slice_gates=circuit.slice_gates,
  figsize_scale=7.5,
)

# How to train a new model from scratch (or finetune an existing one)

In [ ]:
# Generate an untrained model...
allocator = SQARL(
  device=device,
  model_cfg=ModelConfigs(embed_size=64, num_heads=2, num_layers=2),
  mode=SQARL.Mode.Sequential,
)

# ... or load an existing one for finetuning
sqarl = SQARL.load("../trained/da_v2_ft", device=device).set_mode(SQARL.Mode.Sequential) # Load last checkpoint in sequential allocation mode



validation_hardware = Hardware(
  core_capacities=torch.tensor([4]*4),
  core_connectivity=(torch.ones(4,4) - torch.eye(4))
)
val_sampler = RandomCircuit(num_lq=16, num_slices=32)
train_cfg = SQARL.TrainConfig(
  train_iters=50,
  batch_size=1,
  group_size=4,
  validate_each=25,
  validation_hardware=validation_hardware,
  validation_circuits=[val_sampler.sample() for _ in range(32)],
  store_path=f"../trained/test",
  initial_noise=0.2,
  noise_decrease_factor=0.9995,
  min_noise=0.0,
  circ_sampler=RandomCircuit(num_lq=16, num_slices=lambda: randint(4,32)),
  lr=5e-5,
  inv_mov_penalization=0.6,
  mask_invalid=False,
  hardware_sampler=HardwareSampler(max_nqubits=16, range_ncores=[2,8]),
  dropout=0.0,
)
allocator.train(train_cfg)